# Nested feature selection — SVM (RBF kernel)

Leakage-free feature selection for **svm_rbf** on the 2-class ground-type problem. Forward selection
runs **inside each outer leave-user-out fold**, choosing features on an inner grouped CV of only that
fold's training users — so the reported macro-F1 has no selection bias (the honest 'what generalises'
number). Reports per-classifier: honest grouped macro-F1, and which features get selected across folds.

RBF distances are dominated by heavy-tailed features; SUBSAMPLE + max_iter bound cost/heat (O(n^2-3)).

**Thermal/cost guards** (kernel SVMs are O(n^2-3)): `SUBSAMPLE_PER_CLASS` caps train-fold size,
`SVM_MAX_ITER` stops a non-converging fit instead of hanging, `SFS_JOBS=4` uses half the cores.
Every fold prints live progress so the cell never looks frozen.

*Honest limit:* 2-class CV rests on 3 users (soft/hard has 4 users) -> large fold-to-fold variance; treat
gaps under ~3 macro-F1 points as noise. Classifier is UNTUNED (default hyperparameters).

In [1]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from sklearn.svm import SVC

# ================= CONFIG =================
DATA_DIR='..'
MAX_K=15                       # SFS cap (selection plateaus early; keeps runtime bounded)
SFS_TOL=0.0005                  # stop adding features when inner gain < this
SVM_MAX_ITER=50000             # safety cap so a non-converging SVM STOPS, not hangs
SUBSAMPLE_PER_CLASS=2500   # cap per-class train size (kernel SVM cost/heat); None=all
SFS_JOBS=4                     # parallel feature-trials across cores (thermal-safe: half of 8)
CLASS_WEIGHT_STRENGTH=0.5
RANDOM_STATE=42

DROP={'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse','heel_mean_ratio',
      'fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'}|{f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META={'foot','step_id','surface','source_user','source_recording'}

dfs=[]
for fn,lab in [('soft_dry.csv','soft'),('hard_dry.csv','hard')]:
    d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
df=pd.concat(dfs,ignore_index=True)
FEATS=[c for c in df.columns if c not in META and c not in DROP]
classes=sorted(df.surface.unique()); y=df.surface.values; grp=df.source_user.values
_users_per_class=df.groupby('surface')['source_user'].nunique()
N_SPLITS=int(_users_per_class.min())
print(f'{len(df)} steps | {len(FEATS)} features | users/class {_users_per_class.to_dict()} -> N_SPLITS={N_SPLITS}')
print(f'classifier: svm_rbf | SUBSAMPLE_PER_CLASS={SUBSAMPLE_PER_CLASS} | SFS_JOBS={SFS_JOBS}')

16270 steps | 49 features | users/class {'hard': 4, 'soft': 4} -> N_SPLITS=4
classifier: svm_rbf | SUBSAMPLE_PER_CLASS=2500 | SFS_JOBS=4


In [2]:
def _cw_from(ytr):
    if not True: return None
    cs,cnt=np.unique(ytr,return_counts=True); n,k=len(ytr),len(cs)
    return {c:(n/(k*ct))**CLASS_WEIGHT_STRENGTH for c,ct in zip(cs,cnt)}

def make_pipe(ytr):
    _cw=_cw_from(ytr)
    return Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight=_cw, max_iter=SVM_MAX_ITER, random_state=42))])

def _subsample(idx, yv, per_class, seed=RANDOM_STATE):
    if not per_class: return idx
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(yv[idx]):
        ci=idx[yv[idx]==c]; keep+=list(rng.choice(ci,min(len(ci),per_class),replace=False))
    return np.array(sorted(keep))

Xall=df[FEATS].fillna(0).values

def fit_score(cols, tr, te, yv):
    tr=_subsample(tr, yv, SUBSAMPLE_PER_CLASS)
    p=make_pipe(yv[tr]); p.fit(Xall[np.ix_(tr,cols)], yv[tr])
    return f1_score(yv[te], p.predict(Xall[np.ix_(te,cols)]), labels=classes, average='macro')

def inner_cv_score(cols, tr, yv, gv):
    inner=StratifiedGroupKFold(n_splits=max(2,min(N_SPLITS-1,len(np.unique(gv)))),shuffle=True,random_state=0)
    sc=[]
    for itr,ite in inner.split(Xall[tr][:,cols], yv[tr], gv):
        if len(np.unique(yv[tr][itr]))<len(classes): continue
        sub_tr=tr[itr]; sub_te=tr[ite]
        sc.append(fit_score(cols, sub_tr, sub_te, yv))
    return np.mean(sc) if sc else -1

In [3]:
# ===== nested forward selection with LIVE progress =====
splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
_folds=[(tr,te) for tr,te in splitter.split(Xall,y,grp) if len(np.unique(y[tr]))==len(classes)]
print(f'nested forward selection for svm_rbf: {len(_folds)} outer folds, MAX_K={MAX_K}')
outer_mf1=[]; sel_counts={}; per_fold_sets=[]
for _fi,(tr,te) in enumerate(_folds,1):
    t0=time.time(); held=np.unique(grp[te])
    print(f'  fold {_fi}/{len(_folds)} (hold out {held}) — forward-selecting...', flush=True)
    yv,gv=y, grp[tr]
    chosen=[]; remaining=list(range(len(FEATS))); best=-1; step=0
    while remaining and len(chosen)<MAX_K:
        scores=Parallel(n_jobs=SFS_JOBS,prefer='processes')(
            delayed(inner_cv_score)(chosen+[c], tr, yv, gv) for c in remaining)
        s_best,c_best=max(zip(scores,remaining))
        if s_best-best<SFS_TOL: break
        chosen.append(c_best); best=s_best; remaining.remove(c_best); step+=1
        print(f'      + {FEATS[c_best]}  (inner MacroF1 {best*100:.1f}%, {step} feats, {time.time()-t0:.0f}s)', flush=True)
    pf=fit_score(chosen, tr, te, y)
    outer_mf1.append(pf); per_fold_sets.append([FEATS[c] for c in chosen])
    for c in chosen: sel_counts[FEATS[c]]=sel_counts.get(FEATS[c],0)+1
    print(f'    -> fold done: {len(chosen)} feats, held-out MacroF1 {pf*100:.1f}%  ({time.time()-t0:.0f}s)\n', flush=True)
outer_mf1=np.array(outer_mf1)
print(f'=== svm_rbf: nested (leakage-free) grouped MacroF1 {outer_mf1.mean()*100:.1f}% '
      f'(sd {outer_mf1.std()*100:.1f}) | per-fold {np.round(outer_mf1*100,1)} ===')

nested forward selection for svm_rbf: 4 outer folds, MAX_K=15
  fold 1/4 (hold out ['Sam']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 67.2%, 1 feats, 19s)
      + accel_hf_frac  (inner MacroF1 77.0%, 2 feats, 31s)
      + accel_spec_entropy  (inner MacroF1 77.5%, 3 feats, 42s)
      + orient_rest_b_roll3_std  (inner MacroF1 78.0%, 4 feats, 53s)
    -> fold done: 4 feats, held-out MacroF1 66.3%  (64s)

  fold 2/4 (hold out ['Kristian']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 69.4%, 1 feats, 16s)
      + accel_spec_entropy  (inner MacroF1 74.4%, 2 feats, 27s)
      + gyro_x_std  (inner MacroF1 76.4%, 3 feats, 40s)
      + stance_duration_roll3_std  (inner MacroF1 76.4%, 4 feats, 54s)
      + orient_range_b  (inner MacroF1 76.7%, 5 feats, 66s)
      + gyro_hf_frac  (inner MacroF1 76.7%, 6 feats, 78s)
      + orient_rest_b_roll3_std  (inner MacroF1 77.5%, 7 feats, 88s)
      + gu_active_sensors  (inner MacroF1 77.7%, 8 feats, 98s

In [4]:
# ===== selection frequency: which features this classifier relies on across folds =====
freq=sorted(sel_counts.items(), key=lambda kv:-kv[1])
print(f'Feature selection frequency for svm_rbf (across {len(_folds)} folds):')
for f,c in freq: print(f'  {c}/{len(_folds)}  {f}')
print('\nPer-fold selected sets:')
for i,s in enumerate(per_fold_sets,1): print(f'  fold {i}: {s}')
# a 'recommended set' = features chosen in a MAJORITY of folds (stable across users)
maj=[f for f,c in freq if c> len(_folds)//2]
print(f'\nStable set (selected in > half of folds): {maj}')
print('NOTE: with 3 users, per-fold sets vary; the STABLE set is the trustworthy takeaway, '
      'not any single fold\'s pick. Report the nested MacroF1 as this classifier\'s honest accuracy.')

Feature selection frequency for svm_rbf (across 4 folds):
  4/4  orient_rest_a_roll3_std
  3/4  accel_spec_entropy
  3/4  gyro_x_std
  3/4  gu_active_sensors
  2/4  orient_rest_b_roll3_std
  2/4  orient_range_b
  2/4  gyro_hf_frac
  1/4  accel_hf_frac
  1/4  stance_duration_roll3_std
  1/4  heel_impulse_norm
  1/4  gu_stance_frac
  1/4  gyro_z_std
  1/4  time_to_peak_frac

Per-fold selected sets:
  fold 1: ['orient_rest_a_roll3_std', 'accel_hf_frac', 'accel_spec_entropy', 'orient_rest_b_roll3_std']
  fold 2: ['orient_rest_a_roll3_std', 'accel_spec_entropy', 'gyro_x_std', 'stance_duration_roll3_std', 'orient_range_b', 'gyro_hf_frac', 'orient_rest_b_roll3_std', 'gu_active_sensors']
  fold 3: ['orient_rest_a_roll3_std', 'gyro_x_std', 'heel_impulse_norm', 'gu_stance_frac', 'gu_active_sensors']
  fold 4: ['orient_rest_a_roll3_std', 'accel_spec_entropy', 'orient_range_b', 'gyro_x_std', 'gyro_hf_frac', 'gyro_z_std', 'time_to_peak_frac', 'gu_active_sensors']

Stable set (selected in > half of 

## Full-48 baseline vs nested selection

The decisive check (the one missing from the 3-class notebooks): does selection actually beat using **all 48 features**? In 3-class, full-48 won on every metric. This cell reports full-48 under the same grouped CV so you can see immediately whether selection is worthwhile in 2-class.

In [ ]:
# ===== full-48 baseline (the reference nested selection must beat to be worthwhile) =====
# LESSON FROM 3-CLASS: selection looked fine until compared to the FULL feature set, which won.
# Always report this next to the nested number.
from sklearn.metrics import accuracy_score as _acc
def _full_grouped_cv():
    splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
    cols=list(range(len(FEATS))); oof=np.empty(len(y),object); pf=[]
    for tr,te in splitter.split(Xall,y,grp):
        if len(np.unique(y[tr]))<len(classes): continue
        s=fit_score(cols, tr, te, y); pf.append(s)
        # also collect oof for per-class F1
        trs=_subsample(tr,y,SUBSAMPLE_PER_CLASS)
    # recompute oof properly for per-class
    oof=np.empty(len(y),object)
    for tr,te in splitter.split(Xall,y,grp):
        if len(np.unique(y[tr]))<len(classes): continue
        trs=_subsample(tr,y,SUBSAMPLE_PER_CLASS)
        try:
            p=make_pipe(y[trs]) if 'make_pipe' in globals() else None
        except TypeError:
            p=make_pipe()
        if p is None:
            # xgboost path
            m=make_clf(); m.fit(Xall[np.ix_(trs,cols)],_le.transform(y[trs]),sample_weight=_sw(y[trs]))
            oof[te]=_le.inverse_transform(m.predict(Xall[np.ix_(te,cols)]))
        else:
            p.fit(Xall[np.ix_(trs,cols)],y[trs]); oof[te]=p.predict(Xall[np.ix_(te,cols)])
    fill=oof!=None
    mf1=f1_score(y[fill],oof[fill].astype(str),labels=classes,average=None)
    return np.mean(pf), np.std(pf), _acc(y[fill],oof[fill].astype(str)), mf1

_fmean,_fsd,_facc,_fperclass=_full_grouped_cv()
print(f'FULL-48 baseline: Acc {_facc*100:.1f}%  MacroF1 {_fmean*100:.1f}% (sd {_fsd*100:.1f})')
print(f'  per-class F1: ' + '  '.join(f'{c} {v*100:.1f}%' for c,v in zip(classes,_fperclass)))
print(f'\nnested (leakage-free) MacroF1 was {outer_mf1.mean()*100:.1f}% (sd {outer_mf1.std()*100:.1f})')
_gap=(_fmean-outer_mf1.mean())*100
print(f'FULL-48 minus nested-selection: {_gap:+.1f} pts')
if _gap > max(_fsd,outer_mf1.std())*100:
    print('READ: full-48 BEATS selection beyond fold noise -> feature selection HURTS here; use all 48 (as in 3-class).')
elif _gap < -max(_fsd,outer_mf1.std())*100:
    print('READ: selection BEATS full-48 beyond noise -> selection is worthwhile for this classifier in 2-class.')
else:
    print('READ: full-48 and selection are within fold noise -> selection neither helps nor hurts; prefer full-48 for simplicity/robustness.')
